# 03 - Feature Engineering
**Enhancing E-Commerce Sales Forecasting Using Google Trends**

---

## Objectives
1. Select relevant features for modeling
2. Engineer new predictive features
3. Prepare train/test split
4. Scale features
5. Save final feature-engineered dataset

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

df = pd.read_csv('../data/processed/merged_sales_trends.csv')
df['Date'] = pd.to_datetime(df['Date'])
print(f'Loaded: {df.shape}')
df.head()

## 3.1 Feature Selection

We select features based on:
- Domain knowledge (price, season, trends)
- Correlation analysis from EDA
- Avoiding data leakage (no future data)

In [ ]:
# Define feature groups
TREND_KEYWORDS = ['earbuds india', 'wireless earphones', 'bluetooth earbuds',
                   'best earbuds under 2000', 'noise earbuds']

LAG_FEATURES = [f'{kw}_lag{i}' for kw in TREND_KEYWORDS for i in range(1, 5)]

TIME_FEATURES = ['Week_of_Year', 'Month', 'Quarter', 'Is_Festive_Season']

PRICE_FEATURES = ['Avg_Price', 'Weekly_Price', 'Price_Elasticity']

ROLLING_FEATURES = ['Units_Sold_MA2', 'Units_Sold_MA4']

# All features for Model A (baseline - no trends)
BASELINE_FEATURES = TIME_FEATURES + PRICE_FEATURES + ROLLING_FEATURES

# All features for Model B (with Google Trends)
ENHANCED_FEATURES = BASELINE_FEATURES + TREND_KEYWORDS + LAG_FEATURES

TARGET = 'Units_Sold'

print(f'Baseline features: {len(BASELINE_FEATURES)}')
print(f'Enhanced features (with trends): {len(ENHANCED_FEATURES)}')
print(f'\nBaseline: {BASELINE_FEATURES}')

## 3.2 Encode Category

In [ ]:
le = LabelEncoder()
df['Category_Encoded'] = le.fit_transform(df['Category'])

# Add category encoding to both feature sets
BASELINE_FEATURES.append('Category_Encoded')
ENHANCED_FEATURES.append('Category_Encoded')

print('Category mapping:')
for i, cat in enumerate(le.classes_):
    print(f'  {cat} -> {i}')

## 3.3 Add Trend Momentum Feature

In [ ]:
# Trend momentum = current trend - lag2 (is interest increasing or decreasing?)
df['trend_momentum'] = df['earbuds india'] - df['earbuds india_lag2']

# Trend velocity = lag1 - lag3 (rate of change)
df['trend_velocity'] = df['earbuds india_lag1'] - df['earbuds india_lag3']

ENHANCED_FEATURES += ['trend_momentum', 'trend_velocity']

print('Added: trend_momentum, trend_velocity')
print(f'\nFinal enhanced features: {len(ENHANCED_FEATURES)}')

## 3.4 Train / Test Split (Time-Based)

**Important:** We use time-based split, NOT random split.
- Train: 2022-2023
- Test: 2024

In [ ]:
SPLIT_DATE = '2024-01-01'

train_df = df[df['Date'] < SPLIT_DATE].copy()
test_df  = df[df['Date'] >= SPLIT_DATE].copy()

print(f'Train: {train_df.shape}  ({train_df["Date"].min()} to {train_df["Date"].max()})')
print(f'Test:  {test_df.shape}  ({test_df["Date"].min()} to {test_df["Date"].max()})')
print(f'\nSplit ratio: {len(train_df)/(len(train_df)+len(test_df))*100:.1f}% train / {len(test_df)/(len(train_df)+len(test_df))*100:.1f}% test')

In [ ]:
# Prepare X, y for both feature sets
X_train_base = train_df[BASELINE_FEATURES].values
X_test_base  = test_df[BASELINE_FEATURES].values

X_train_enh  = train_df[ENHANCED_FEATURES].values
X_test_enh   = test_df[ENHANCED_FEATURES].values

y_train = train_df[TARGET].values
y_test  = test_df[TARGET].values

print(f'X_train_base: {X_train_base.shape}  |  X_train_enhanced: {X_train_enh.shape}')
print(f'X_test_base:  {X_test_base.shape}  |  X_test_enhanced:  {X_test_enh.shape}')
print(f'y_train: {y_train.shape}  |  y_test: {y_test.shape}')

## 3.5 Feature Scaling

In [ ]:
scaler_base = StandardScaler()
X_train_base_s = scaler_base.fit_transform(X_train_base)
X_test_base_s  = scaler_base.transform(X_test_base)

scaler_enh = StandardScaler()
X_train_enh_s = scaler_enh.fit_transform(X_train_enh)
X_test_enh_s  = scaler_enh.transform(X_test_enh)

print('Feature scaling done (StandardScaler)')
print(f'Scaled means close to 0: {X_train_enh_s.mean(axis=0)[:5].round(2)}')

## 3.6 Save Prepared Data

In [ ]:
import pickle

# Save feature-engineered DataFrames
train_df.to_csv('../data/processed/train_data.csv', index=False)
test_df.to_csv('../data/processed/test_data.csv', index=False)

# Save feature lists and scalers
artifacts = {
    'BASELINE_FEATURES': BASELINE_FEATURES,
    'ENHANCED_FEATURES': ENHANCED_FEATURES,
    'TARGET': TARGET,
    'scaler_base': scaler_base,
    'scaler_enh': scaler_enh,
    'label_encoder': le,
    'SPLIT_DATE': SPLIT_DATE
}

with open('../data/processed/feature_artifacts.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print('Saved:')
print('  data/processed/train_data.csv')
print('  data/processed/test_data.csv')
print('  data/processed/feature_artifacts.pkl')
print(f'\nReady for modeling (Notebook 04)!')

## 3.7 Feature Importance Preview (Correlation with Target)

In [ ]:
corrs = train_df[ENHANCED_FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['tab:green' if v > 0 else 'tab:red' for v in corrs.values]
corrs.plot(kind='barh', color=colors, ax=ax, edgecolor='black')
ax.set_xlabel('Correlation with Units_Sold')
ax.set_title('Feature Correlation with Sales', fontsize=14, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print(f'\nTop 5 positive:')
for feat, val in corrs.head(5).items():
    print(f'  {feat:35s}  r = {val:.3f}')

print(f'\nTop 5 negative:')
for feat, val in corrs.tail(5).items():
    print(f'  {feat:35s}  r = {val:.3f}')

---

## Summary

| Feature Set | # Features | Description |
|---|---|---|
| Baseline | ~10 | Time + Price + Rolling averages (NO trends) |
| Enhanced | ~33 | Baseline + Google Trends + Lags + Momentum |

**Research Hypothesis:** Enhanced model (with Google Trends) will outperform baseline model.

**Next:** `04_sarima_baseline.ipynb` for SARIMA time-series baseline

---